# Wi-Fi CSI human sensing — reproducing the paper models and building a presence / motion detector

**Goal of the project:** detect whether a person is in a room — and whether they are still or moving — from the Wi-Fi Channel State Information (CSI) produced by our own ESP32 pipeline.

**Goal of this notebook:**

1. Re-implement, in PyTorch, the human-activity-recognition (HAR) models from the papers in `../papers`, with the **LSTM-CNN of Shang et al. (2021) as the core model** and the other papers' models as extra comparisons.
2. Re-implement every pre-processing step used in those papers as a documented, reusable Python class (`csi_preprocessing.py`).
3. Train and evaluate all models on the public HAR dataset of Zhuravchak et al. (`data/wifi_csi_har_dataset`, downloaded with `python download_data.py`), both the way the papers did it **and** in a stricter way that measures how well a model works on recordings and rooms it has never seen.
4. Apply the pipeline to our own ESP32 recordings (`data/self_extracted_dataset`), show what works and what the current data cannot tell us yet, and give concrete recommendations for collecting better data.

### Contents
1. [The papers and the decisions taken](#1)
2. [Setup](#2)
3. [The HAR dataset](#3)
4. [Pre-processing classes, step by step](#4)
5. [Windows and evaluation protocols](#5)
6. [Core model: LSTM-CNN (Shang et al. 2021)](#6)
7. [Extra models: BiLSTM, InceptionTime, PA-CSI](#7)
8. [Results overview](#8)
9. [Our own ESP32 data](#9)
10. [Conclusions and recommendations](#10)
11. [Reproducibility](#11)

### How to use this notebook
| File | Content |
|---|---|
| `csi_preprocessing.py` | all pre-processing steps as scikit-learn-style classes (`fit` / `transform`) |
| `csi_data.py` | loading, label mappings per task, per-paper feature recipes, splits, lazy window datasets |
| `csi_models.py` | PyTorch models (LSTM-CNN, LSTM, BiLSTM, InceptionTime, PA-CSI) |
| `csi_train.py` | training loop, metrics, result cache |
| `csi_experiments.py` | `run_har(model, task, protocol)` — one experiment end-to-end; all hyper-parameters in `MODEL_SPECS` |
| `run_all.py` | runs the whole experiment grid in parallel processes (≈ 3–4 h on a 20-core CPU, no GPU needed) |
| `csi_esp32.py` | ESP32-specific pipeline (parsing, resampling, blocked cross-validation) |
| `results/*.json`, `models/*.pt` | cached metrics / weights of every experiment |

Every experiment cell calls `run_har(...)`. If the result is already in `results/` it is loaded in a second; otherwise the model is trained. Set `FORCE = True` in the setup cell to retrain everything from scratch inside the notebook (slow), or run `python run_all.py` in a terminal first.

<a id="1"></a>
## 1. The papers and the decisions taken

| Paper | Type | What we take from it |
|---|---|---|
| **Shang et al. 2021**, *LSTM-CNN network for HAR using WiFi CSI data* | model | **Core model: LSTM-CNN** (LSTM 28 units → Conv2d 5 × (30×28) → max-pool → MLP), LSTM baseline, KNN / SVM baselines, Butterworth low-pass de-noising, 2- vs 4-channel input (amplitude product + antenna phase difference [+ both amplitudes]), binary static/dynamic task |
| **Zhuravchak et al. 2022**, *HAR based on Wi-Fi CSI data — a deep neural network approach* | dataset + models | **The dataset we train on**; phase sanitisation, Hampel outlier removal, wavelet de-noising; windows of 1024 packets labelled by their last packet; **BiLSTM** and **InceptionTime** |
| **Quy et al. 2025**, *PA-CSI* (Sensors 25, 1038) | model | Kalman filter, phase unwrapping, time alignment, min-max normalisation; **PA-CSI**: Gaussian Range Encoding + Multi-scale Convolution Augmented Transformer layers on amplitude and phase + Gated Residual Network fusion; ablations |
| **Guarino et al. 2026**, *survey on CSI datasets and models — reproducibility* | survey | Methodology: random splits of overlapping windows inflate scores; evaluate across sessions/environments; report everything needed to reproduce |
| **Korany & Mostofi 2021**, *nocturnal seizure detection with WiFi* | health | Hampel filter, PCA de-noising, motion detection from spectral energy above the breathing band — used on our ESP32 data |
| **Chang et al. 2023**, *ALESAL* (sleep apnea / limb movement) | health | Resampling of irregular packets, per-packet min-max normalisation, band-pass filter, PCA — used for the ESP32 pipeline |

### Key decisions (and why)

| Decision | Choice | Why |
|---|---|---|
| Training data | Zhuravchak HAR dataset (10 recordings, 3 rooms, 67 k packets, 7 activities) | The only labelled dataset available; it contains "no person", static and moving activities, i.e. everything the end goal needs. |
| Tasks | `activity7` (paper task), **`presence_motion3`** (empty / static / moving = end goal), `static_dynamic2` (Shang's binary task) | `activity7` lets us compare with the papers; `presence_motion3` is what we actually want for our room and matches the labels of our ESP32 recordings. |
| Label of a window | label of the **last packet** of the window | As in Zhuravchak et al. — simulates a real-time system that classifies "now" from the recent past. |
| Common evaluation grid | every model predicts the same target packets (one every 16 packets) | Each model uses its own window length (400 / 1000 / 1024), but scores are computed on identical targets, so they are comparable. |
| Evaluation protocols | `random` (as in the papers) **and** `cross_session` / `cross_room` | Overlapping windows in a random split are near-copies of each other, so `random` mostly measures memorisation. Held-out sessions/rooms measure what we care about: working in a new recording or a new room. |
| Class imbalance | class-weighted cross-entropy; report **macro-F1** next to accuracy | `walking` is 39 % of the data, `get_up`/`get_down` ≈ 4 % each — plain accuracy rewards ignoring the rare classes. |
| Model selection | best validation macro-F1 (early stopping); the test set is never used for decisions | Standard practice; prevents tuning on the test set. |
| Hardware | CPU only (20 cores, no GPU) | Forced some reductions for the extra models (listed per model in section 7); **the core LSTM-CNN is implemented exactly at paper size.** |

<a id="2"></a>
## 2. Setup

**Environment.** Python 3.12 with PyTorch, scikit-learn, SciPy, PyWavelets, pandas, matplotlib (see `pyproject.toml`). Training runs on the best available device, chosen by `csi_train.get_device()`: an **Intel GPU** via PyTorch's `xpu` backend (this machine has an Intel Arc A370M), an NVIDIA GPU (`cuda`), or the CPU. Set the environment variable `CSI_DEVICE=cpu` to force the CPU. All random seeds are fixed in `csi_train.set_seed`.

In [ ]:
import os, glob, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import csi_preprocessing as pp
import csi_data as cd
import csi_models as cm
import csi_plots as cp
from csi_experiments import run_har, get_recordings, get_features, prepare, MODEL_SPECS
from csi_train import train_torch_model, predict, metrics, set_seed, DEVICE

torch.set_num_threads(8)
cp.setup_style()
pd.set_option("display.precision", 3)
FORCE = False          # True = retrain every experiment instead of loading results/*.json
print("torch", torch.__version__, "| device", DEVICE, "| threads", torch.get_num_threads(), "| cwd", os.getcwd())

<a id="3"></a>
## 3. The HAR dataset (Zhuravchak et al. 2022)

**Recording setup (paper, Table 1):** two TP-Link TL-WDR4300 routers with the Atheros CSI tool, 5 GHz, channel 60, 40 MHz bandwidth, 2 Tx × 2 Rx antennas → 4 antenna pairs × 114 subcarriers. One person performs walking, sitting, standing, lying, getting up, getting down — or leaves the room ("no_person") — in no fixed order, in 3 rooms. Labels come from a camera image taken with every packet.

**File layout (found by inspection, not documented):** `data.csv` has no header and 1026 columns:

| columns | content |
|---|---|
| 0 – 113 | the numbers 0…113 in every row (subcarrier index — dropped) |
| 114 – 569 | amplitude, 4 antenna pairs × 114 subcarriers (pair-major) |
| 570 – 1025 | phase in [−π, π], same order |

`data.csv` has exactly one row more than `label.csv`; labels are aligned from the first row and the extra last row is dropped. The parsed arrays are cached in `cache/`.

**No timestamps** are stored, so the packet rate is unknown. This matters for the filters: all cut-off frequencies on this dataset are therefore given as a fraction of the Nyquist frequency instead of in Hz.

In [ ]:
recs = get_recordings()
rows = []
for r in recs:
    vc = pd.Series(r.labels).value_counts()
    rows.append({"recording": r.name, "packets": len(r), **vc.to_dict()})
overview = pd.DataFrame(rows).fillna(0).set_index("recording")[["packets"] + cd.ACTIVITIES].astype(int)
overview.loc["total"] = overview.sum()
overview

In [ ]:
tot = overview.loc["total", cd.ACTIVITIES].sort_values()
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh(tot.index, tot.values, color=cp.PALETTE[0], height=0.6)
for i, v in enumerate(tot.values):
    ax.text(v + 200, i, f"{v:,}  ({100 * v / tot.sum():.0f} %)", va="center", fontsize=8, color=cp.MUTED)
ax.set_xlim(0, tot.max() * 1.25); ax.grid(axis="y", visible=False)
ax.set_title("Packets per activity (all recordings)", loc="left"); ax.set_xlabel("packets")
plt.show()

The data is **strongly imbalanced**: walking dominates, the transitions `get_down` / `get_up` are rare (and short by nature). This is why we use a class-weighted loss and report macro-F1.

Below: one recording as a heat-map (amplitude of antenna pair 0, subcarriers × time) with the activity labels on top. Movement (walking, getting up/down) shows up as fast fluctuations over time; static activities and the empty room give a stable pattern whose *level and shape* depend on where the person is.

In [ ]:
r = recs[0]
codes = cd.encode_labels(r.labels, "activity7")
fig, (a0, a1) = plt.subplots(2, 1, figsize=(11, 4.2), sharex=True, gridspec_kw={"height_ratios": [1, 4]})
a0.imshow(codes[None], aspect="auto", cmap=plt.matplotlib.colors.ListedColormap(cp.PALETTE[:7]),
          vmin=-0.5, vmax=6.5, interpolation="nearest"); a0.set_yticks([]); a0.grid(False)
a0.set_title(f"{r.name}: activity labels (top) and amplitude of antenna pair 0 (bottom)", loc="left")
handles = [plt.matplotlib.patches.Patch(color=cp.PALETTE[i], label=a) for i, a in enumerate(cd.ACTIVITIES)]
a0.legend(handles=handles, ncol=7, loc="lower left", bbox_to_anchor=(0, 1.25), fontsize=7)
a1.imshow(r.amp[:, :114].T, aspect="auto", cmap="Blues", interpolation="nearest"); a1.grid(False)
a1.set_ylabel("subcarrier"); a1.set_xlabel("packet index")
plt.show()

<a id="4"></a>
## 4. Pre-processing classes, step by step

All steps live in **`csi_preprocessing.py`**. Each is a class with the scikit-learn transformer interface (`fit`, `transform`, `fit_transform`), so that

* steps can be chained in a `sklearn.pipeline.Pipeline`,
* the *same fitted object* can later be applied to live ESP32 data,
* stateful steps (min-max ranges, PCA axes, standardisation) are fitted on **training data only**, which prevents information from the test set leaking into the model.

Where a well-established library already implements the maths, the class is a thin, documented wrapper around it:

| Class | Paper | What it does | Implementation |
|---|---|---|---|
| `HampelFilter` | Zhuravchak, Korany | replaces outliers (> 3 robust σ from the rolling median) by the median | `scipy.ndimage.median_filter` |
| `WaveletDenoiser` | Zhuravchak | DWT, soft-threshold detail coefficients (VisuShrink), inverse DWT | `pywt` |
| `PhaseSanitizer` | Zhuravchak (Sen et al.) | removes the linear phase error (CFO/SFO) across subcarriers | NumPy |
| `PhaseUnwrapper` | PA-CSI | removes 2π jumps along time or subcarriers | `numpy.unwrap` |
| `PhaseDifference` | Shang, Korany | phase difference between two antennas (cancels CFO) | NumPy |
| `AmplitudeProduct` | Shang | product of two antennas' amplitudes | NumPy |
| `ButterworthFilter` | Shang, ALESAL | low/high/band-pass IIR filter, zero-phase or causal | `scipy.signal.butter`, `sosfiltfilt` |
| `KalmanSmoother` | PA-CSI | random-walk Kalman filter per stream (optional RTS smoother) | NumPy |
| `TimeAligner` | PA-CSI | averages fixed-size bins → fixed sequence length | NumPy |
| `MinMaxNormalizer` | PA-CSI (global), ALESAL (per packet) | scales to [0, 1] | `sklearn.preprocessing.MinMaxScaler` |
| `Standardizer` | — | z-score per stream | `sklearn.preprocessing.StandardScaler` |
| `PCADenoiser` | Korany, ALESAL | first principal component(s) per antenna pair | `sklearn.decomposition.PCA` |
| `SubcarrierSelector` | — | keep a subset of subcarriers | NumPy |
| `ESP32CSIParser`, `UniformResampler` | ALESAL (resampling) | our ESP32 CSV format → uniform time series | pandas, `numpy.interp` |

Every class has a docstring with the paper equation it implements — e.g.:

In [ ]:
help(pp.PhaseSanitizer)

In the following cells each step is applied to a real stream of recording `room_1/1` (packets 1500–3000, which contain walking, standing and sitting) so you can see what it does.

### 4.1 Hampel outlier removal (Zhuravchak 2022, Korany 2021)
**Why:** commodity NICs produce isolated spikes (rate adaptation, power control, thermal noise) that are not caused by people. A median-based filter removes them without blurring real edges, unlike a moving average. Window ±5 packets, threshold 3 robust standard deviations — the textbook defaults.

In [ ]:
seg = slice(1500, 3000)
x = r.amp[seg, 40]                                       # pair 0, subcarrier 40
h = pp.HampelFilter(window=5, n_sigma=3)
xh = h.transform(r.amp[seg])[:, 40]
cp.before_after(x, xh, f"Hampel filter — {100 * h.outlier_fraction_:.1f} % of all samples replaced")
plt.show()

### 4.2 Wavelet de-noising (Zhuravchak 2022)
**Why:** CSI noise is broadband while human motion is concentrated in a few large wavelet coefficients. Shrinking the small coefficients removes noise but keeps sharp transitions (sitting down, standing up) better than a low-pass filter would. We use Daubechies-4, 4 levels, the universal threshold σ√(2 ln n) with σ estimated from the finest detail level (Donoho & Johnstone) — the paper does not give its parameters, these are the standard choices.

In [ ]:
xw = pp.WaveletDenoiser("db4", level=4).transform(xh)
cp.before_after(xh, xw, "Wavelet (db4, VisuShrink soft threshold) after Hampel"); plt.show()

### 4.3 Phase sanitisation (Sen et al. 2012, used by Zhuravchak 2022)
**Why:** the raw phase contains a random offset per packet (carrier frequency offset, CFO) and a slope across subcarriers (sampling frequency offset / packet detection delay). Both change from packet to packet and are much larger than the phase change caused by a person, so the raw phase is useless. Unwrapping across subcarriers and subtracting the best line `a·m_k + b` (m_k = physical subcarrier index −58…58) removes both errors and keeps the shape that the multipath channel — and therefore the person — imposes.

Left: phase across subcarriers for 5 consecutive packets (raw = scattered, sanitised = consistent). Right: one subcarrier over time.

In [ ]:
san = pp.PhaseSanitizer(n_subcarriers=114, subcarrier_indices=pp.ht40_subcarrier_indices())
ph_s = san.transform(r.phase[seg])
fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))
for k in range(5):
    ax[0].plot(r.phase[seg][k, :114], color=cp.PALETTE[0], alpha=0.5, lw=1)
    ax[0].plot(ph_s[k, :114], color=cp.PALETTE[1], lw=1.4)
ax[0].set_title("across subcarriers: raw (blue) vs sanitised (orange)", loc="left"); ax[0].set_xlabel("subcarrier")
cp.before_after(r.phase[seg, 40], ph_s[:, 40], "one subcarrier over time", ax=ax[1], labels=("raw", "sanitised"))
plt.tight_layout(); plt.show()

### 4.4 Phase unwrapping along time (Quy 2025, PA-CSI)
**Why (paper):** the phase is only known modulo 2π; unwrapping makes it continuous so a network can follow it over time (PA-CSI Eq. 7).

**What we found:** on this dataset, unwrapping the **raw** phase along time produces a random walk (the per-packet CFO jumps are up to ±π, so `numpy.unwrap` cannot tell real jumps from wrap-arounds). Unwrapping the **sanitised** phase gives a bounded, meaningful signal. The PA-CSI recipe therefore sanitises first — a documented deviation (PA-CSI was developed on Intel 5300 data). Because unwrapped phase can still drift slowly, it is centred per window (`center_cols` in `WindowDataset`).

In [ ]:
uw_raw = pp.PhaseUnwrapper("time", remove_mean=True).transform(r.phase[seg, 40:41])[:, 0]
uw_san = pp.PhaseUnwrapper("time", remove_mean=True).transform(ph_s[:, 40:41])[:, 0]
cp.before_after(uw_raw, uw_san, "time-unwrapped phase: raw (random walk) vs sanitised", labels=("raw, unwrapped", "sanitised, unwrapped"))
plt.show()

### 4.5 Antenna phase difference and amplitude product (Shang 2021)
**Why:** both receive antennas share the same oscillator, so the CFO/SFO errors are (almost) identical on both and cancel in the **difference** of their phases — a cheap alternative to sanitisation that is widely used for breathing and motion sensing (also by Korany 2021). Shang et al. combine it with the **product** of the two amplitudes ("2-channel input") and optionally the two amplitudes themselves ("4-channel input", better in their Fig. 4).

**Assumption:** the column order of the 4 antenna pairs is not documented; we take pairs (0, 1) and (2, 3) as "same Tx antenna, Rx antenna 1 vs 2". The plot shows the difference is far more stable than a single raw phase.

In [ ]:
pdiff = pp.PhaseDifference(((0, 1), (2, 3)), 114).transform(r.phase[seg])
prod = pp.AmplitudeProduct(((0, 1), (2, 3)), 114).transform(r.amp[seg])
fig, ax = plt.subplots(1, 2, figsize=(11, 2.6))
cp.before_after(r.phase[seg, 40], pdiff[:, 40], "phase: raw antenna 1 vs difference ant1 − ant2", ax=ax[0], labels=("raw phase", "phase difference"))
ax[1].plot(prod[:, 40], color=cp.PALETTE[0]); ax[1].set_title("amplitude product |H1|·|H2|", loc="left"); ax[1].set_xlabel("packet index")
plt.tight_layout(); plt.show()

### 4.6 Butterworth low-pass filter (Shang 2021)
**Why:** human motion changes the channel slowly compared with the packet rate, so most high-frequency content is noise. A 4th-order Butterworth filter is maximally flat in the pass-band (it does not distort the motion band). We use `sosfiltfilt` (forward + backward → zero phase delay, good for offline training); `causal=True` gives the real-time variant. Cut-off 0.1 × Nyquist because the dataset has no timestamps (paper: not specified).

In [ ]:
xb = pp.ButterworthFilter(cutoff=0.1, btype="low", order=4).transform(x)
xbc = pp.ButterworthFilter(cutoff=0.1, btype="low", order=4, causal=True).transform(x)
ax = cp.before_after(x, xb, "Butterworth low-pass (0.1 × Nyquist): zero-phase vs causal", labels=("raw", "zero-phase (offline)"))
ax.plot(xbc, color=cp.PALETTE[2], lw=1.2, label="causal (real-time)"); ax.legend(ncol=3, loc="upper right"); plt.show()

### 4.7 Kalman filter (Quy 2025, PA-CSI)
**Why:** the Kalman filter is the optimal linear estimator of a hidden state from noisy measurements under Gaussian noise. PA-CSI does not specify the state model; we use the simplest one for a slowly varying amplitude (random walk + white measurement noise), with the measurement noise estimated from the data and process noise = 5 % of it. The forward filter is causal (usable live); the optional Rauch-Tung-Striebel pass (`rts=True`) smooths offline.

In [ ]:
xk = pp.KalmanSmoother(process_ratio=0.05).transform(x)
xks = pp.KalmanSmoother(process_ratio=0.05, rts=True).transform(x)
ax = cp.before_after(x, xk, "Kalman filter (causal) and RTS smoother", labels=("raw", "Kalman (causal)"))
ax.plot(xks, color=cp.PALETTE[2], lw=1.2, label="Kalman + RTS"); ax.legend(ncol=3, loc="upper right"); plt.show()

### 4.8 Time alignment (Quy 2025, PA-CSI)
**Why:** PA-CSI maps recordings of unequal length to a fixed length by averaging equal bins (their Fig. 6). We use the same operation to shrink a 1000-packet window to 100 steps (bins of 10 packets) — it acts as low-pass filter + down-sampling and makes the attention layers affordable on a CPU.

In [ ]:
w = r.amp[2000:3000, 40]
wa = pp.TimeAligner(100).transform(w)
fig, ax = plt.subplots(figsize=(9, 2.4))
ax.plot(np.arange(1000), w, color=cp.PALETTE[0], alpha=0.5, lw=1, label="1000 packets")
ax.plot(np.arange(100) * 10 + 5, wa, "o-", ms=3, color=cp.PALETTE[1], label="aligned to 100 steps")
ax.legend(); ax.set_title("Time alignment by bin averaging", loc="left"); plt.show()

### 4.9 Normalisation (Quy 2025 global min-max, Chang 2023 per-packet min-max, z-score)
* **Global min-max / z-score** (fitted on training recordings): puts all features on a comparable scale for gradient descent.
* **Per-packet min-max** (ALESAL Eq. 3): every packet is scaled by its own min and max over the subcarriers of one antenna pair. This removes changes of the overall gain (automatic gain control) and keeps only the *shape* across frequency. It is essential for the ESP32 data (section 9), where the absolute level differs between recordings for reasons unrelated to the person.

In [ ]:
pk = pp.MinMaxNormalizer("per_sample", n_subcarriers=114).transform(r.amp[seg])
fig, ax = plt.subplots(1, 2, figsize=(11, 2.6))
ax[0].imshow(r.amp[seg, :114].T, aspect="auto", cmap="Blues"); ax[0].grid(False); ax[0].set_title("raw amplitude (pair 0)", loc="left")
ax[1].imshow(pk[:, :114].T, aspect="auto", cmap="Blues"); ax[1].grid(False); ax[1].set_title("per-packet min-max", loc="left")
for a in ax: a.set_xlabel("packet"); a.set_ylabel("subcarrier")
plt.tight_layout(); plt.show()

### 4.10 PCA de-noising (Korany 2021, Chang 2023)
**Why:** a person moving changes all subcarriers in a correlated way, while noise is independent per subcarrier. The first principal component across subcarriers therefore concentrates the motion and averages out the noise. Korany et al. build their seizure/motion detector on exactly this signal; we use it for the ESP32 motion detector. Below, the first PC of antenna pair 0 for the whole recording, coloured by activity — movement periods stand out.

In [ ]:
pca = pp.PCADenoiser(n_components=1, n_subcarriers=114).fit(r.amp)
pc = pca.transform(r.amp)[:, 0]
fig, ax = plt.subplots(figsize=(11, 2.6))
for i, a in enumerate(cd.ACTIVITIES):
    m = codes == i
    ax.plot(np.flatnonzero(m), pc[m], ".", ms=1.5, color=cp.PALETTE[i], label=a)
ax.legend(ncol=7, markerscale=6, loc="upper left", fontsize=7); ax.set_title("First principal component, antenna pair 0", loc="left")
ax.set_xlabel("packet index"); plt.show()

### 4.11 Chaining steps in a scikit-learn `Pipeline`
Because every class follows the transformer API, a pre-processing chain is one object that can be stored (e.g. with `joblib`) and applied unchanged to new data:

In [ ]:
from sklearn.pipeline import Pipeline
amp_pipeline = Pipeline([
    ("hampel", pp.HampelFilter(window=5, n_sigma=3)),
    ("wavelet", pp.WaveletDenoiser("db4", level=4)),
    ("scale", pp.Standardizer()),
])
out = amp_pipeline.fit_transform(r.amp)
print(amp_pipeline, "\n->", out.shape, out.mean().round(3), out.std().round(3))

### 4.12 The per-paper recipes
Each model is trained on the pre-processing of *its own* paper (functions `recipe_*` in `csi_data.py`, cached in `cache/`):

| Recipe | Used by | Steps | Features per packet |
|---|---|---|---|
| `shang4` | LSTM-CNN, LSTM, KNN, SVM | Butterworth low-pass on amplitude and on antenna phase difference → [ampₐ, amp_b, ampₐ·amp_b, Δphase] for 2 antenna groups × 114 subcarriers | 912 |
| `shang2` | LSTM-CNN (2-channel) | as above, only [ampₐ·amp_b, Δphase] | 456 |
| `zhuravchak` | BiLSTM, InceptionTime | amplitude: Hampel → wavelet; phase: sanitise → Hampel → wavelet | 912 |
| `pacsi` | PA-CSI | amplitude: Kalman (+ global min-max); phase: sanitise → unwrap in time (+ per-window centring); every 4th subcarrier | 232 |

All recipes except PA-CSI are followed by a z-score fitted on the training recordings only.

In [ ]:
for name in ["shang4", "shang2", "zhuravchak", "pacsi"]:
    f = get_features(name)
    print(f"{name:11s} -> {len(f)} recordings, features per packet: {f[0].shape[1]}")

<a id="5"></a>
## 5. Windows and evaluation protocols

**Windows.** A model sees the last `L` packets and predicts the activity of the last packet (`L` = 400 for Shang, 1024 for Zhuravchak, 1000 for PA-CSI — the paper values). Targets are taken every 16 packets, starting at packet 1023 of each recording so that every model has a full window; windows never cross recordings. Windows are cut on the fly (`WindowDataset`) — materialising all overlapping 1024 × 912 windows would need ≈ 30 GB.

**Protocols.**

| Protocol | Train | Validation | Test | Measures |
|---|---|---|---|---|
| `random` | 70 % of all targets | 15 % | 15 % | what the papers report (leaky, see below) |
| `cross_session` | 6 recordings | room_1/3, room_3/4 | **room_1/4, room_3/5** | new recording of a known room |
| `cross_room` | 7 recordings (rooms 1, 3) | room_1/4, room_3/5 | **room_2/1** (room never seen) | a new room — closest to deploying in *our* room |

**Why `random` is leaky.** Neighbouring targets are 16 packets apart, but windows are 400–1024 packets long, so a random test window shares 96–98 % of its packets with a training window. The cell below measures this.

In [ ]:
targets = cd.target_grid(recs, 16, 1024)
sp = cd.split_targets(targets, recs, "random", seed=0)
train_set = {(int(a), int(b)) for a, b in sp["train"]}
dist = []
for rr, e in sp["test"]:
    d = min(abs(e - e2) for (r2, e2) in train_set if r2 == rr)
    dist.append(d)
dist = np.array(dist)
print(f"median distance test target -> nearest training target: {np.median(dist):.0f} packets")
for L in (400, 1000, 1024):
    print(f"  window {L:4d}: median overlap with a training window = {100 * np.median(np.clip(1 - dist / L, 0, 1)):.1f} %")

In [ ]:
rows = []
for protocol in ["random", "cross_session", "cross_room"]:
    sp = cd.split_targets(targets, recs, protocol, seed=0)
    for part, t in sp.items():
        y = np.array([cd.encode_labels(np.array([recs[a].labels[b]]), "presence_motion3")[0] for a, b in t])
        rows.append({"protocol": protocol, "split": part, "windows": len(t),
                     **{c: int((y == i).sum()) for i, c in enumerate(cd.TASKS["presence_motion3"]["classes"])}})
pd.DataFrame(rows).set_index(["protocol", "split"])

<a id="6"></a>
## 6. Core model: LSTM-CNN (Shang et al. 2021)

### 6.1 Architecture
Shang et al. argue that LSTM and CNN are complementary: the LSTM extracts temporal features from the signal fluctuations, the CNN then finds the most informative *stretch* of that feature sequence. Implementation (`csi_models.ShangLSTMCNN`), with the paper's sizes:

| Step | Paper | Shape for one 400-packet window |
|---|---|---|
| input (4-channel, 2 antenna groups × 114 subcarriers) | Sec. 2.1 | 400 × 912 |
| LSTM, 28 hidden units, dropout 0.2 | Sec. 2.2.1 | 400 × 28 |
| "dimensional promotion" to one image channel | Sec. 2.2.1 | 1 × 400 × 28 |
| Conv2d, 5 kernels of 30 × 28, stride 1, ReLU | Sec. 2.2.2, Eq. 7 | 5 × 371 × 1 |
| max-pool over time (one value per kernel) | Eq. 8 | 5 |
| MLP classifier (size not given → 64 hidden units, dropout 0.2) | Sec. 2.2.2 | n_classes |

The model is tiny (≈ 0.1 M parameters, almost all in the LSTM input weights), which is attractive for a small, possibly embedded detector. The 30 × 28 kernel spans *all* LSTM units, so each kernel is a learned 30-step temporal template over the LSTM state, and the max-pool asks "did this template occur anywhere in the window?".

In [ ]:
net = cm.ShangLSTMCNN(n_features=912, n_classes=3)
print(net)
print("trainable parameters:", cm.count_parameters(net))
x = torch.randn(2, 400, 912)
h, _ = net.lstm(x); q = h.unsqueeze(1); c = net.conv(q)
print("input", tuple(x.shape), "-> LSTM", tuple(h.shape), "-> Q", tuple(q.shape), "-> conv", tuple(c.shape), "-> logits", tuple(net(x).shape))

### 6.2 Training setup
| Setting | Value | Source / reason |
|---|---|---|
| optimiser | Adam, learning rate 0.001 | Shang Sec. 3.2 |
| batch size | 10 | Shang Sec. 3.2 |
| window length | 400 packets | Shang Sec. 2.2.1 ("400-length sequence") |
| epochs | up to 60, early stopping (patience 12) on validation macro-F1 | paper: 50–70 iterations; early stopping avoids picking the epoch by hand |
| windows per epoch | random 3000 of the training windows (re-drawn every epoch) | keeps an epoch at ≈ 30 s on the CPU |
| loss | cross-entropy **with class weights** | deviation: our data is imbalanced; Shang's was roughly balanced |
| input scaling | z-score per feature, fitted on the training recordings | standard for LSTMs |

The full training function is `csi_train.train_torch_model`. First the end-goal task on held-out sessions:

In [ ]:
res = run_har("lstm_cnn", "presence_motion3", "cross_session", force=FORCE)
h = pd.DataFrame(res["history"])
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
ax[0].plot(h.epoch, h.loss, color=cp.PALETTE[0]); ax[0].set_title("training loss", loc="left"); ax[0].set_xlabel("epoch")
ax[1].plot(h.epoch, h.val_acc, color=cp.PALETTE[0], label="accuracy"); ax[1].plot(h.epoch, h.val_f1, color=cp.PALETTE[1], label="macro-F1")
ax[1].axvline(res["best_epoch"], color=cp.MUTED, ls="--", lw=1); ax[1].legend(); ax[1].set_title("validation", loc="left"); ax[1].set_xlabel("epoch")
cp.confusion(res["test"]["confusion"], res["classes"], "test (room_1/4 + room_3/5), recall %", ax=ax[2])
plt.tight_layout(); plt.show()
print({k: round(v, 3) for k, v in res["test"].items() if k not in ("confusion", "n")})

### 6.3 LSTM-CNN on all tasks and protocols

In [ ]:
settings = [("activity7", "random"), ("activity7", "cross_session"), ("presence_motion3", "random"),
            ("presence_motion3", "cross_session"), ("presence_motion3", "cross_room")]
runs = {s: run_har("lstm_cnn", *s, force=FORCE, verbose=False) for s in settings}
pd.DataFrame([{"task": t, "protocol": p, "accuracy": r_["test"]["accuracy"], "macro_F1": r_["test"]["f1_macro"],
               "best_epoch": r_["best_epoch"], "train_min": r_["train_time_s"] / 60} for (t, p), r_ in runs.items()])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
cp.confusion(runs[("activity7", "random")]["test"]["confusion"], cd.ACTIVITIES, "activity7 — random split", ax=ax[0])
cp.confusion(runs[("activity7", "cross_session")]["test"]["confusion"], cd.ACTIVITIES, "activity7 — held-out sessions", ax=ax[1])
cp.confusion(runs[("presence_motion3", "cross_room")]["test"]["confusion"], cd.TASKS["presence_motion3"]["classes"], "presence_motion3 — unseen room", ax=ax[2])
plt.tight_layout(); plt.show()

### 6.4 Shang's own comparison: LSTM vs LSTM-CNN, 2 vs 4 channels, KNN / SVM
Shang et al. report (on their own 5-activity dataset, which is not public): LSTM-CNN 94.1 % vs LSTM 82.7 %, KNN 74.1 %, SVM 71.3 % (multi-class) and 95.0 / 89.8 / 84.3 / 77.0 % on the binary static/dynamic task; 4-channel input better than 2-channel. We repeat the comparison on the binary `static_dynamic2` task (empty-room packets excluded, as in their data) and on `activity7`, with held-out sessions.

**KNN / SVM features** (the paper does not list them): for each of the 8 channel groups the subcarrier-averaged series is summarised by mean, std, min, max, range, mean |first difference|, skewness, kurtosis and mean per-subcarrier std → 72 features; standardised; KNN with k = 5, RBF-SVM with C = 10 and balanced class weights.

In [ ]:
rows = []
for model in ["knn", "svm", "lstm", "lstm_cnn_2ch", "lstm_cnn"]:
    for task in ["static_dynamic2", "activity7"]:
        if task == "activity7" and model in ("knn", "svm", "lstm"):
            continue   # covered in section 8
        r_ = run_har(model, task, "cross_session", force=FORCE, verbose=False)
        rows.append({"model": model, "task": task, "accuracy": r_["test"]["accuracy"], "macro_F1": r_["test"]["f1_macro"]})
shang_tab = pd.DataFrame(rows)
paper = {("knn", "static_dynamic2"): 0.843, ("svm", "static_dynamic2"): 0.770, ("lstm", "static_dynamic2"): 0.898, ("lstm_cnn", "static_dynamic2"): 0.950}
shang_tab["paper_accuracy (their data)"] = [paper.get((m, t), np.nan) for m, t in zip(shang_tab.model, shang_tab.task)]
shang_tab

In [ ]:
sd = shang_tab[shang_tab.task == "static_dynamic2"].melt(id_vars="model", value_vars=["accuracy", "macro_F1"], var_name="metric")
cp.grouped_bars(sd, "value", "model", "metric", title="static vs dynamic, held-out sessions")
plt.show()

<!-- DISCUSSION_6 -->

<a id="7"></a>
## 7. Extra models

The models below come from the other papers. They are included to see whether a larger or more recent architecture beats the LSTM-CNN. Where the CPU forced a reduction, it is listed — all reductions are in `csi_experiments.MODEL_SPECS`.

### 7.1 BiLSTM (Zhuravchak 2022)
One bidirectional LSTM layer with 256 hidden units, followed by dense layers 512 → 256 → 128 → classes (ReLU). The 1024-packet window is processed "in 16-time-point chunks with 8-point overlap": each chunk (16 × 912 values) is one LSTM step → 127 steps. Paper result: 61.1 % validation accuracy (7 classes) — with heavy overfitting.

| Deviation | Why |
|---|---|
| a linear layer maps every 14 592-value chunk to 256 values before the LSTM | the faithful model has 31 M LSTM parameters and needs 0.7 s per window per training step on our CPU (≈ 30 min per epoch); with the projection 10 ms. The projection is also a strong regulariser against the overfitting the paper reports. |
| dropout 0.3 between dense layers, weight decay 1e-4 | the paper reports severe overfitting |

### 7.2 InceptionTime (Ismail Fawaz et al. 2020, used by Zhuravchak 2022)
Six inception modules (bottleneck 32 channels, parallel convolutions of length 39/19/9 + a max-pool branch, 32 filters each), residual connection every three modules, global average pooling, linear output. Paper result: 38.2 % validation accuracy, 99.2 % training accuracy — strong overfitting.

| Deviation | Why |
|---|---|
| window averaged from 1024 to 256 steps (`TimeAligner`) | 8× cheaper (180 → 22 ms per window); kernels still cover 40 × 4 = 160 packets |
| one network instead of an ensemble of five | CPU time; an ensemble would add a few points at most |

### 7.3 PA-CSI (Quy et al. 2025)
Amplitude and phase are processed separately, each by a **temporal stream** (tokens = time steps; attention over time) and a **channel stream** (tokens = subcarrier streams, i.e. the transposed input; attention across subcarriers). Each stream has 4 **MCAT layers** (multi-head self-attention + multi-scale CNN, residual + LayerNorm). **Gaussian Range Encoding** (K = 10 learnable Gaussians over time, multiplied with a learnable embedding) adds order information to the temporal tokens. Each stream is summarised by "CNN max-pooling" (multi-kernel 1-D convolutions + global max), and a **Gated Residual Network** (ELU dense layers, LayerNorm, sigmoid gate, residual from the amplitude branch) fuses amplitude and phase into 256 units. Paper results: 98–99.9 % on StanWiFi, MultiEnv and their own lab dataset (random 80/20 split or 5-fold CV).

| Deviation | Why |
|---|---|
| window 1000 → 100 time steps (bins of 10 packets) | attention cost; paper used 1000–2000 steps on a GPU |
| every 4th subcarrier: 4 pairs × 29 = 116 amplitude + 116 phase streams | paper inputs had 90 streams (Intel 5300); 456 streams per modality made the channel attention 30× slower |
| phase sanitised before unwrapping | see section 4.4 |
| no dropout on the attention weights; batch 16; 40 epochs (decay 0.95 / epoch) instead of 200 | CPU time (dropout on attention maps was 40 % of the run time) |
| temporal width 96 (8 heads), channel width = 100 time steps (10 heads) | head counts must divide the token widths (paper: 9 and 50 heads for 90 × 1000 inputs) |

In [ ]:
net = cm.PACSI(seq_len=100, n_amp=116, n_phase=116, n_classes=7)
print("PA-CSI parameters:", cm.count_parameters(net))
print("BiLSTM parameters:", cm.count_parameters(cm.ZhuravchakBiLSTM(912, 7, input_proj=256)),
      "| faithful BiLSTM:", cm.count_parameters(cm.ZhuravchakBiLSTM(912, 7)))
print("InceptionTime parameters:", cm.count_parameters(cm.InceptionTime(912, 7)))

In [ ]:
rows = []
for model in ["bilstm", "inceptiontime", "pacsi"]:
    for task, protocol in [("activity7", "random"), ("activity7", "cross_session"),
                           ("presence_motion3", "cross_session"), ("presence_motion3", "cross_room")]:
        r_ = run_har(model, task, protocol, force=FORCE, verbose=False)
        rows.append({"model": model, "task": task, "protocol": protocol, "accuracy": r_["test"]["accuracy"],
                     "macro_F1": r_["test"]["f1_macro"], "best_epoch": r_["best_epoch"], "train_min": r_["train_time_s"] / 60})
pd.DataFrame(rows)

### 7.4 PA-CSI ablations
Quy et al. claim that (i) combining phase with amplitude, (ii) the Gaussian Range Encoding and (iii) the GRN fusion each improve accuracy. We switch each component off (activity7, held-out sessions):

In [ ]:
rows = []
for model in ["pacsi", "pacsi_amp_only", "pacsi_phase_only", "pacsi_no_gre", "pacsi_no_grn"]:
    r_ = run_har(model, "activity7", "cross_session", force=FORCE, verbose=False)
    rows.append({"variant": model, "accuracy": r_["test"]["accuracy"], "macro_F1": r_["test"]["f1_macro"]})
abl = pd.DataFrame(rows); abl

<!-- DISCUSSION_7 -->

<a id="8"></a>
## 8. Results overview

All cached results are collected into one table. **Accuracy** is comparable with the papers; **macro-F1** (mean F1 over classes) is the fairer number under class imbalance.

In [ ]:
res_all = []
for f in sorted(glob.glob("results/har__*.json")):
    d = json.load(open(f))
    res_all.append({"model": d["model"], "task": d["task"], "protocol": d["protocol"], "seed": d["seed"],
                    "accuracy": d["test"]["accuracy"], "macro_F1": d["test"]["f1_macro"],
                    "val_macro_F1": d["val"]["f1_macro"], "n_test": d["test"]["n"]})
res_all = pd.DataFrame(res_all)
main = res_all[(res_all.seed == 0) & res_all.model.isin(["knn", "svm", "lstm", "lstm_cnn", "bilstm", "inceptiontime", "pacsi"])]
order = ["knn", "svm", "lstm", "lstm_cnn", "bilstm", "inceptiontime", "pacsi"]
piv = main.pivot_table(index="model", columns=["task", "protocol"], values=["accuracy", "macro_F1"]).reindex(order)
piv.round(3)

### 8.1 Reproducing the papers (7 activities): random split vs held-out sessions
Paper numbers for reference: Zhuravchak BiLSTM 61.1 % (validation) / 87.8 % ("testing"), InceptionTime 38.2 %; Shang LSTM-CNN 94.1 % and PA-CSI 98–99.9 % on *other* datasets.

In [ ]:
a7 = main[main.task == "activity7"].copy()
a7["protocol"] = a7.protocol.map({"random": "random split (paper style)", "cross_session": "held-out sessions"})
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
cp.grouped_bars(a7, "accuracy", "model", "protocol", "activity7 — accuracy", ax=ax[0])
cp.grouped_bars(a7, "macro_F1", "model", "protocol", "activity7 — macro-F1", ax=ax[1])
plt.tight_layout(); plt.show()

### 8.2 The end goal (empty / static / moving): new session and new room

In [ ]:
pm = main[(main.task == "presence_motion3") & (main.protocol != "random")].copy()
pm["protocol"] = pm.protocol.map({"cross_session": "held-out sessions", "cross_room": "unseen room (room_2)"})
fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
cp.grouped_bars(pm, "accuracy", "model", "protocol", "presence_motion3 — accuracy", ax=ax[0])
cp.grouped_bars(pm, "macro_F1", "model", "protocol", "presence_motion3 — macro-F1", ax=ax[1])
plt.tight_layout(); plt.show()

### 8.3 How stable is the core model? (3 seeds)
Neural-network results vary with the random initialisation. Differences between models smaller than this spread are not meaningful.

In [ ]:
seeds = res_all[(res_all.model == "lstm_cnn") & (res_all.task == "presence_motion3") & (res_all.protocol == "cross_session")]
print(seeds[["seed", "accuracy", "macro_F1"]].to_string(index=False))
print(f"accuracy {seeds.accuracy.mean():.3f} ± {seeds.accuracy.std():.3f},  macro-F1 {seeds.macro_F1.mean():.3f} ± {seeds.macro_F1.std():.3f}")

### 8.4 Predictions over time in the unseen room
The plot shows, for the LSTM-CNN and the best extra model, the predicted class for every target of room_2 (never seen in training) against the ground truth. It shows *where* the errors happen (transitions, long static periods, ...).

In [ ]:
best_extra = pm[pm.protocol.str.startswith("unseen") & (pm.model != "lstm_cnn")].sort_values("macro_F1").model.iloc[-1]
fig, ax = plt.subplots(2, 1, figsize=(12, 4.6), sharex=True)
for a, m in zip(ax, ["lstm_cnn", best_extra]):
    d = run_har(m, "presence_motion3", "cross_room", verbose=False)
    cp.label_timeline(d["test_true"], d["test_pred"], d["classes"], f"{m} on room_2/1 (macro-F1 {d['test']['f1_macro']:.2f})", ax=a)
plt.tight_layout(); plt.show()

<!-- DISCUSSION_8 -->

<a id="9"></a>
## 9. Our own ESP32 data

`data/self_extracted_dataset` contains three 2-minute recordings made with our ESP32 pipeline: `empty_room`, `standing_still`, `moving` — exactly the three classes of the end goal.

### 9.1 What the capture contains
The hardware is very different from the HAR dataset: 2.4 GHz (channel 6), 20 MHz, **one** antenna, 64 CSI entries per packet (52 usable subcarriers), and no controlled transmitter. Inspection shows that the receiver logged **every** device on the channel:

In [ ]:
import csi_esp32 as ce
raw = {c: pd.read_csv(f"data/self_extracted_dataset/{f}") for c, f in ce.ESP32_FILES.items()}
rows = []
for c, df in raw.items():
    top = df.mac.value_counts()
    t = df[df.mac == top.index[0]].timestamp.to_numpy()
    rows.append({"recording": c, "packets": len(df), "distinct MACs": df.mac.nunique(), "main MAC": top.index[0],
                 "main MAC packets": top.iloc[0], "main MAC rate [Hz]": len(t) / (t[-1] - t[0]),
                 "median gap [s]": np.median(np.diff(t)), "max gap [s]": np.diff(t).max(),
                 "packets with 128 entries": int((df.subcarrier_count == 128).sum())})
pd.DataFrame(rows).set_index("recording")

**Consequences for the pipeline:**
* CSI from different transmitters describes different radio links (different positions, hardware, gain) and must not be mixed → `ESP32CSIParser` keeps only the main MAC (`D2:ED:D8:7D:ED:EB`, the access point).
* That link delivers only ≈ 3–4 packets/s, irregularly (gaps up to 2 s) → `UniformResampler` interpolates onto a uniform **5 Hz** grid (ALESAL) and flags gaps > 1 s. 5 Hz is enough to see body movement (< 2.5 Hz) but far below the HAR dataset's rate, so models trained there cannot be applied directly.
* Packets with 128 entries (HT-LTF included) are rare → dropped; entries 0 and 27–37 are always zero (DC and guard subcarriers) → dropped, leaving 52 subcarriers.

### 9.2 A shortcut in the data: the received signal strength

In [ ]:
recs32 = ce.load_esp32("data/self_extracted_dataset", fs=5.0)
fig, ax = plt.subplots(1, 2, figsize=(13, 3))
for k, rc in enumerate(recs32):
    ax[0].plot(rc.t - rc.t[0], rc.rssi, color=cp.PALETTE[k], label=ce.ESP32_CLASSES[k])
    ax[1].plot(rc.t - rc.t[0], rc.amp.mean(1), color=cp.PALETTE[k], label=ce.ESP32_CLASSES[k])
ax[0].set_title("RSSI of the main link [dBm]", loc="left"); ax[1].set_title("mean CSI amplitude (52 subcarriers)", loc="left")
for a in ax: a.set_xlabel("time in recording [s]"); a.legend(ncol=3)
plt.tight_layout(); plt.show()

The RSSI is −29 dBm in the empty room and −41 / −43 dBm with a person present — a 12 dB step with almost no overlap. **A classifier given the RSSI separates the classes perfectly without learning anything about people.** The step may be real (a person blocking the line of sight) or may come from something else that changed between recordings (a device moved, a door, the router's transmit power). With one recording per class we cannot tell. The CSI amplitude itself is held almost constant by the ESP32's automatic gain control.

**Decision:** RSSI and absolute amplitude are **not** used as model inputs. Every packet is min-max normalised across its subcarriers (`MinMaxNormalizer("per_sample")`, ALESAL), so models only see the *shape* of the channel across frequency and how it changes over time. We still report an RSSI-only classifier to make the shortcut visible.

### 9.3 Pre-processed data

In [ ]:
feats32 = {m: [ce.normalise(rc.amp, m) for rc in recs32] for m in ("raw", "per_packet")}
fig, ax = plt.subplots(1, 3, figsize=(14, 2.8))
for k, rc in enumerate(recs32):
    ax[k].imshow(feats32["per_packet"][k].T, aspect="auto", cmap="Blues", interpolation="nearest"); ax[k].grid(False)
    ax[k].set_title(f"{ce.ESP32_CLASSES[k]} — per-packet normalised", loc="left"); ax[k].set_xlabel("sample (5 Hz)")
ax[0].set_ylabel("subcarrier"); plt.tight_layout(); plt.show()

### 9.4 Evaluation: blocked 5-fold cross-validation over time
Each class is a single continuous recording, so a random split would put almost identical overlapping windows in train and test. Instead each recording is cut into 5 consecutive blocks of 24 s; in fold *k*, block *k* of every recording is the test set, and training windows may not overlap it. Every window is tested exactly once.

Windows are **5 s (25 samples)** long. Models:
* **RSSI only** — logistic regression on mean/std of the RSSI (the shortcut, for reference).
* **SVM** on 6 window statistics, on raw and on per-packet-normalised amplitude.
* **LSTM-CNN** (Shang) on the 52 per-packet-normalised amplitudes; the CNN kernel is shortened to 10 steps (the window has 25), everything else as in section 6. No validation set exists inside a fold, so it is trained for a fixed 25 epochs (selection on the training data, never on the test block).
* **Motion energy** (Korany): spectral energy of the first principal component above 0.6 Hz (twice the maximal breathing rate). Unsupervised — it can only tell *moving* from *not moving*.

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

W, FS = 25, 5.0
y_of = lambda tg: np.array([recs32[a].label for a, _ in tg])
pred = {k: [] for k in ["RSSI only (logreg)", "SVM, raw amplitude", "SVM, per-packet", "LSTM-CNN, per-packet"]}
y_all = []
set_seed(0)
for fold, tr, te in ce.blocked_folds(recs32, W, k=5):
    ytr, yte = y_of(tr), y_of(te); y_all += list(yte)
    rssi_f = lambda tg: np.array([[recs32[a].rssi[b - W + 1:b + 1].mean(), recs32[a].rssi[b - W + 1:b + 1].std()] for a, b in tg])
    pred["RSSI only (logreg)"] += list(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(rssi_f(tr), ytr).predict(rssi_f(te)))
    for mode, key in [("raw", "SVM, raw amplitude"), ("per_packet", "SVM, per-packet")]:
        Xtr, Xte = ce.window_features(feats32[mode], tr, W), ce.window_features(feats32[mode], te, W)
        pred[key] += list(make_pipeline(StandardScaler(), SVC(class_weight="balanced")).fit(Xtr, ytr).predict(Xte))
    # LSTM-CNN: z-score fitted on the training windows' samples only
    f = feats32["per_packet"]
    frames = np.concatenate([f[a][b - W + 1:b + 1] for a, b in tr[::W]])
    mu, sd = frames.mean(0), frames.std(0) + 1e-6
    fz = [((x - mu) / sd).astype(np.float32) for x in f]
    dtr, dte = ce.ESP32WindowDataset(fz, recs32, tr, W), ce.ESP32WindowDataset(fz, recs32, te, W)
    net = cm.ShangLSTMCNN(52, 3, kernel_time=10)
    train_torch_model(net, dtr, dtr, 3, epochs=25, batch_size=16, patience=100, seed=fold, verbose=False)
    pred["LSTM-CNN, per-packet"] += list(predict(net, dte))
    print(f"fold {fold}: train {len(tr)} windows, test {len(te)} windows")
esp_res = pd.DataFrame([{"model": k, **{m: v for m, v in metrics(y_all, p, 3).items() if m in ("accuracy", "f1_macro")}} for k, p in pred.items()])
esp_res

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
for a, (k, p) in zip(ax, pred.items()):
    cp.confusion(metrics(y_all, p, 3)["confusion"], ce.ESP32_CLASSES, k, ax=a)
plt.tight_layout(); plt.show()

### 9.5 The unsupervised motion detector (Korany 2021)

In [ ]:
all_t = np.array([(a, e) for a in range(3) for e in range(W - 1, len(recs32[a].t))])
me = ce.motion_energy(feats32["per_packet"], all_t, W, FS, f_cut=0.6)
lab = y_of(all_t)
fig, ax = plt.subplots(figsize=(8, 2.8))
bins = np.logspace(np.log10(me.min()), np.log10(me.max()), 50)
for k in range(3):
    ax.hist(me[lab == k], bins=bins, color=cp.PALETTE[k], alpha=0.6, label=ce.ESP32_CLASSES[k])
ax.set_xscale("log"); ax.legend(); ax.set_xlabel("spectral energy above 0.6 Hz of the 1st PC (log)")
ax.set_title("Motion energy per 5 s window", loc="left")
# threshold calibrated on the first 20 % of the empty-room recording only (like Korany's calibration period)
calib = me[(lab == 0) & (all_t[:, 1] < 0.2 * len(recs32[0].t))]
thr = 3 * calib.max()
ax.axvline(thr, color=cp.MUTED, ls="--", lw=1); plt.show()
moving_pred = me > thr
print(f"threshold = 3 x max of the calibration period = {thr:.3f}")
for k in range(3):
    print(f"  {ce.ESP32_CLASSES[k]:7s}: {100 * moving_pred[lab == k].mean():5.1f} % of windows flagged as 'moving'")

<!-- DISCUSSION_9 -->

### 9.6 Using the pipeline on a new recording
The whole chain — parse → resample → per-packet normalisation → windows → model — can be applied to any new CSV written by the ESP32 pipeline. Below, an LSTM-CNN is trained on *all* windows of the three recordings and then applied to a file (here, for illustration, again `moving.csv`; in practice a new recording). This is the starting point for a live detector.

In [ ]:
def esp32_file_to_windows(path, fs=5.0, window=W, mac="D2:ED:D8:7D:ED:EB"):
    parser = pp.ESP32CSIParser(mac=mac)
    t, amp, _, _ = parser.to_arrays(parser.parse(path))
    grid, a = pp.UniformResampler(fs=fs).transform_with_time(t, amp)
    a = pp.MinMaxNormalizer("per_sample", a.shape[1]).transform(a)
    a = ((a - mu_all) / sd_all).astype(np.float32)
    idx = np.arange(window - 1, len(a))
    return grid[idx] - grid[0], torch.from_numpy(np.stack([a[e - window + 1:e + 1] for e in idx]))

f = feats32["per_packet"]; frames = np.concatenate(f); mu_all, sd_all = frames.mean(0), frames.std(0) + 1e-6
fz = [((x - mu_all) / sd_all).astype(np.float32) for x in f]
ds_all = ce.ESP32WindowDataset(fz, recs32, all_t, W)
detector = cm.ShangLSTMCNN(52, 3, kernel_time=10)
train_torch_model(detector, ds_all, ds_all, 3, epochs=25, batch_size=16, patience=100, verbose=False)

tt, Xw = esp32_file_to_windows("data/self_extracted_dataset/moving.csv")
with torch.no_grad():
    p = detector.eval()(Xw.to(DEVICE)).softmax(-1).cpu().numpy()
fig, ax = plt.subplots(figsize=(10, 2.4))
for k in range(3):
    ax.plot(tt, p[:, k], color=cp.PALETTE[k], label=ce.ESP32_CLASSES[k])
ax.set_xlabel("time [s]"); ax.set_ylabel("probability"); ax.legend(ncol=3); ax.set_title("Detector output on a recording", loc="left"); plt.show()

<a id="10"></a>
## 10. Conclusions and recommendations

<!-- CONCLUSIONS -->

### Recommended data-collection protocol for the ESP32
The models are ready; what limits a robust detector for our room is the **data**. Recommended changes, in order of impact:

1. **Use a dedicated transmitter instead of sniffing.** Flash one ESP32 with `esp-csi/examples/get-started/csi_send` and one with `csi_recv` (or `csi_recv_router` pinging the router). This gives one known link at a fixed, high rate (aim for **≥ 50–100 packets/s**) instead of ≈ 3 Hz from whatever the access point happens to send. Filter on the transmitter MAC in the receiver.
2. **Many short sessions instead of one long one per class.** For example 10–20 sessions of 2–5 min per class, spread over **several days and times of day**, with the door open/closed, lights/devices on/off. Only then can a model learn "person" rather than "the room at 19:52".
3. **Record transitions and mixed sessions**, e.g. enter → stand → sit → walk → leave in one recording, with a timestamped label log (a small script where you press a key at each change, or a phone video synchronised by timestamp). Real use is continuous, not one activity per file.
4. **Vary the person and the position:** several people, different positions in the room (near and far from the line of sight), different speeds.
5. **Keep the hardware fixed** (positions of Tx and Rx marked on the floor, antenna orientation, channel) and log it in a metadata file per session — and if possible add a **second receiver** (more antennas help when the person is not in the line of sight; Zhuravchak experiment 4).
6. **Store raw data** (timestamp, MAC, RSSI, real/imaginary parts, as now) and **never discard empty-room recordings**: periodic empty-room recordings are the reference for recalibration (Korany's calibration period).

Evaluate new data with **leave-one-session-out** or **leave-one-day-out** cross-validation (`blocked_folds` generalises easily), never with random window splits.

<a id="11"></a>
## 11. Reproducibility
* All experiment hyper-parameters: `csi_experiments.MODEL_SPECS`; tasks and splits: `csi_data.TASKS`, `csi_data.SPLITS`.
* Re-run everything: delete `results/` and `models/` and run `python run_all.py` (≈ 3–4 h, 5 processes × 4 threads), then re-run this notebook; or set `FORCE = True`.
* Seeds are fixed (`seed=0` unless stated); PyTorch CPU kernels are deterministic for these models up to floating-point summation order.

In [ ]:
import sklearn, scipy, pywt, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__,
      "| sklearn", sklearn.__version__, "| pywt", pywt.__version__, "| pandas", pd.__version__)
print("experiments in results/:", len(glob.glob("results/har__*.json")))